In [ ]:
# Download the full Tabula Sapiens dataset directly from CellXGene.
# Source: https://datasets.cellxgene.cziscience.com/5a495302-b7cd-4bf9-853e-95627b00bb03.h5ad
import os

CXG_DOWNLOAD_URL = "https://datasets.cellxgene.cziscience.com/5a495302-b7cd-4bf9-853e-95627b00bb03.h5ad"
adata_path = "tabula_sapiens_full.h5ad"

In [7]:
# Restart the runtime before running this cell
import anndata
import numpy as np
import scanpy as sc
import scvi

/home/cane/.local/share/hatch/env/virtual/scvi-tools/eVVa01t5/scvi-tools/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/cane/.local/share/hatch/env/virtual/scvi-tools/eVVa01t5/scvi-tools/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/cane/.local/share/hatch/env/virtual/scvi-tools/eVVa01t5/scvi-tools/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/cane/.local/share/hatch/env/virtual/scvi-tools/eVVa01t5/scvi-tools/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndat

In [8]:
import gc
import os

In [ ]:
# Download the full Tabula Sapiens h5ad from CellXGene (skips if already cached).
import urllib.request

if not os.path.exists(adata_path):
    print(f"Downloading Tabula Sapiens dataset to {adata_path} ...")
    urllib.request.urlretrieve(CXG_DOWNLOAD_URL, adata_path)
    print("Download complete.")
else:
    print(f"Using cached dataset at {adata_path}.")

adata_full = anndata.read_h5ad(adata_path)
print(adata_full)

In [ ]:
# Inspect available tissues in the dataset.
# The CellXGene h5ad uses 'tissue_in_publication' with PascalCase values
# that match the config names directly (e.g. 'Bone_Marrow', 'Bladder').
tissue_column = "tissue_in_publication"
print("Available tissues:")
print(sorted(adata_full.obs[tissue_column].unique()))

We loop here through all tissues and train all 4 models on the input data.
The full Tabula Sapiens dataset is loaded **once** from CellXGene, then subsetted per tissue.
For each tissue we apply HVG filtering and train: scVI, scANVI (from scVI), CondSCVI, Stereoscope.
scVI and scANVI are minified (latent representations stored in obsm) before saving.
CondSCVI and Stereoscope are saved without anndata (no minification support).
We use non-default parameters for scVI/scANVI to make the models compatible with scArches.
3 layers are used to account for stronger batch effects due to SmartSeq2 and 10X data.

In [ ]:
# Parameters for Tabula Sapiens training
# NOTE: verify these column names against adata_full.obs.columns after inspecting the dataset.
tissue_column = "tissue_in_publication"  # obs column with tissue labels (PascalCase, e.g. 'Bone_Marrow')
labels_key = "cell_type"                 # obs column with cell type annotations
batch_key = "donor_assay"               # obs column for batch correction (donor × assay)
unknown_celltype_label = "unknown"

# Folder to store trained models
model_dir = "pretrained_model_final/"

# Tissues to train — values match tissue_in_publication exactly.
tissues = [
    "Bladder", "Blood", "Bone_Marrow", "Ear", "Eye", "Fat", "Heart", "Kidney",
    "Large_Intestine", "Liver", "Lung", "Lymph_Node", "Mammary", "Muscle",
    "Ovary", "Pancreas", "Prostate", "Salivary_Gland", "Skin", "Small_Intestine",
    "Spleen", "Stomach", "Testis", "Thymus", "Tongue", "Trachea", "Uterus",
    "Vasculature",
]

scvi_model_kwargs = {
    "dropout_rate": 0.05,
    "dispersion": "gene",
    "n_layers": 3,
    "n_latent": 20,
    "gene_likelihood": "nb",
    "use_batch_norm": "none",
    "use_layer_norm": "both",
    "encode_covariates": True,
}

In [10]:
finished = [] # 'Tongue', 'Mammary', 'Salivary_Gland', 'Skin', 'Heart'

In [ ]:
for tissue in tissues:
    if tissue in finished:
        print(f"{tissue} already finished. Skipping.")
        continue

    # tissue_in_publication values match the config names directly (no conversion needed).
    tissue_adata = adata_full[adata_full.obs[tissue_column] == tissue].copy()
    print(f"\n=== {tissue}: {tissue_adata.n_obs} cells ===")

    if tissue_adata.n_obs == 0:
        print(f"WARNING: no cells found for tissue '{tissue}'. Available: {sorted(adata_full.obs[tissue_column].unique())}")
        continue

    sc.pp.highly_variable_genes(
        tissue_adata,
        n_top_genes=3000,
        subset=True,
        flavor="seurat_v3",
        span=1.0,
        batch_key=batch_key,
    )

    # ── scVI ──────────────────────────────────────────────────────────────
    print(f"  [{tissue}] Training SCVI …")
    scvi.model.SCVI.setup_anndata(tissue_adata, batch_key=batch_key, labels_key=labels_key)
    scvi_model = scvi.model.SCVI(tissue_adata, **scvi_model_kwargs)
    scvi_model.train(max_epochs=100, train_size=0.8, plan_kwargs={"n_epochs_kl_warmup": 100})

    # ── scANVI (initialised from scVI before minification) ────────────────
    print(f"  [{tissue}] Training SCANVI …")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model, unlabeled_category=unknown_celltype_label
    )
    scanvi_model.train(max_epochs=20, n_samples_per_label=20, plan_kwargs={"n_epochs_kl_warmup": 10})

    # Minify and save scVI
    qzm, qzv = scvi_model.get_latent_representation(give_mean=False, return_dist=True)
    scvi_model.adata.obsm["scvi_latent_qzm"] = qzm
    scvi_model.adata.obsm["scvi_latent_qzv"] = qzv
    scvi_model.minify_adata(use_latent_qzm_key="scvi_latent_qzm", use_latent_qzv_key="scvi_latent_qzv")
    scvi_model.save(f"{model_dir}{tissue}/scvi", overwrite=True, save_anndata=True)

    # Minify and save scANVI
    qzm, qzv = scanvi_model.get_latent_representation(give_mean=False, return_dist=True)
    scanvi_model.adata.obsm["scanvi_latent_qzm"] = qzm
    scanvi_model.adata.obsm["scanvi_latent_qzv"] = qzv
    scanvi_model.minify_adata(use_latent_qzm_key="scanvi_latent_qzm", use_latent_qzv_key="scanvi_latent_qzv")
    scanvi_model.save(f"{model_dir}{tissue}/scanvi", overwrite=True, save_anndata=True)

    # ── CondSCVI (fresh setup on original tissue_adata copy) ──────────────
    print(f"  [{tissue}] Training CondSCVI …")
    condscvi_adata = tissue_adata.copy()
    scvi.model.CondSCVI.setup_anndata(condscvi_adata, labels_key=labels_key)
    condscvi_model = scvi.model.CondSCVI(
        condscvi_adata, n_latent=5, n_layers=2, dropout_rate=0.05, weight_obs=False
    )
    condscvi_model.train(max_epochs=200, train_size=0.8)
    condscvi_model.save(f"{model_dir}{tissue}/condscvi", overwrite=True, save_anndata=False)

    # ── Stereoscope (fresh setup on original tissue_adata copy) ──────────
    print(f"  [{tissue}] Training Stereoscope …")
    stereo_adata = tissue_adata.copy()
    scvi.external.RNAStereoscope.setup_anndata(stereo_adata, labels_key=labels_key)
    stereoscope_model = scvi.external.RNAStereoscope(stereo_adata)
    stereoscope_model.train(max_epochs=100, train_size=0.8)
    stereoscope_model.save(f"{model_dir}{tissue}/stereoscope", overwrite=True, save_anndata=False)

    gc.collect()
    finished.append(tissue)
    print(f"{tissue} done.")

# Models are saved locally in model_dir.
# To upload to Hugging Face Hub, run the automated workflow:
#   python -m scvi_hub_models --model_name tabula_sapiens --reload_data True --reload_model True